In [ ]:
import os

# change working directory so that it picks up the grapehelper library
os.chdir("/home/ftorgano/rna-kg-analysis")
print(os.getcwd())

In [ ]:
import importlib
import logging

import helper_lib.graph
import helper_lib.cache
import helper_lib.predict

importlib.reload(helper_lib)
importlib.reload(helper_lib.graph)
importlib.reload(helper_lib.cache)
importlib.reload(helper_lib.predict)
helper_lib.cache.set_embedding_cache_dir(
    "./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/"
)
logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
import pandas as pd

In [ ]:
view_number = 9

In [ ]:
view_undirected_rnakg = helper_lib.graph.load_view_rnakg(view_number, directed=False)

In [ ]:
view_undirected_rnakg

In [ ]:
df_view = helper_lib.graph.build_triples_df(view_undirected_rnakg)

# Running predictions

## LINE

In [ ]:
from grape.embedders import FirstOrderLINEEnsmallen
from grape.edge_prediction import DecisionTreeEdgePrediction, RandomForestEdgePrediction

seed = 42

model_tree = DecisionTreeEdgePrediction(
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    random_state=seed,
)
model_forest = RandomForestEdgePrediction(
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    n_estimators=100,
)
embedder_line = FirstOrderLINEEnsmallen(
    random_state=seed, enable_cache=False, embedding_size=10, verbose=False
)

In [ ]:
import importlib

importlib.reload(helper_lib.predict)

# miRNA-Phenotype fails to generate negative train/test set
pairs_to_predict = [
    ("miRNA", "Phenotype"),
    ("lncRNA", "Phenotype"),
    ("miRNA", "Gene"),
    ("miRNA", "GO"),
    ("lncRNA", "GO"),
    ("lncRNA", "Disease"),
    ("Protein", "GO"),
]

results_fun_line_tree = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_line,
    model_tree,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    use_scale_free_distribution=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
)
results_fun_line_tree.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_tree_f1_2.csv"
)
results_fun_line_forest = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_line,
    model_forest,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    use_scale_free_distribution=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
)
results_fun_line_forest.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_forest_f1_2.csv"
)

full_results = pd.concat([results_fun_line_tree, results_fun_line_forest])
full_results.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_line_f1_2.csv"
)

# results: results biased because the negative training set can have edges in common with the negative test set
# results_f1: results fixed by generating the negative training set and avoiding the edges in the negative test set
# results_f1_2: results fixed by generating a big negative graph and splitting it into train/test sets like in HetNode2Vec work

In [ ]:
full_results = pd.read_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_line_f1_2.csv"
)
for elem in full_results["name"].unique():
    for pair in pairs_to_predict:
        df_results = full_results[full_results["name"] == elem]
        model = "".join(df_results["Model"].iloc[0].split(" ")[:2])
        results_custom_model = df_results[
            (df_results["Source Type"] == pair[0])
            & (df_results["Destination Type"] == pair[1])
        ]
        positive = results_custom_model["Positive balanced accuracy"]
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%", "\\%")
        negative = results_custom_model["Negative balanced accuracy"]
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%", "\\%")
        mean = results_custom_model["Mean balanced accuracy"]
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%", "\\%")
        f1 = results_custom_model["F1-Binary"]
        f1_print = f"{f1.mean():.2%}±{f1.std():.2%}".replace("%", "\\%")
        print(
            f"{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} & {f1_print} \\\\"
        )

### Unbalanced

In [ ]:
results_fun_line_tree_unbalanced = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_line,
    model_tree,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
    testing_unbalance_rate=10,
)
results_fun_line_tree_unbalanced.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_tree_f1_2_unbalanced.csv"
)
results_fun_line_forest_unbalanced = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_line,
    model_forest,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
    testing_unbalance_rate=10,
)
results_fun_line_forest_unbalanced.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_line_forest_f1_2_unbalanced.csv"
)

full_results_unabalanced = pd.concat(
    [results_fun_line_tree_unbalanced, results_fun_line_forest_unbalanced]
)
full_results_unabalanced.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_line_f1_2_unbalanced.csv"
)

In [ ]:
full_results_unabalanced = pd.read_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_line_f1_2_unbalanced.csv"
)

for elem in full_results_unabalanced["name"].unique():
    for pair in pairs_to_predict:
        df_results = full_results_unabalanced[full_results_unabalanced["name"] == elem]
        model = "".join(df_results["Model"].iloc[0].split(" ")[:2])
        results_custom_model = df_results[
            (df_results["Source Type"] == pair[0])
            & (df_results["Destination Type"] == pair[1])
        ]
        positive = results_custom_model["Positive balanced accuracy"]
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%", "\\%")
        negative = results_custom_model["Negative balanced accuracy"]
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%", "\\%")
        mean = results_custom_model["Mean balanced accuracy"]
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%", "\\%")
        f1 = results_custom_model["F1-Binary"]
        f1_print = f"{f1.mean():.2%}±{f1.std():.2%}".replace("%", "\\%")
        print(
            f"{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} & {f1_print}\\\\"
        )

## Node2Vec SkipGram BFS

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
from grape.edge_prediction import DecisionTreeEdgePrediction, RandomForestEdgePrediction

seed = 42

model_tree = DecisionTreeEdgePrediction(
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    random_state=seed,
)
model_forest = RandomForestEdgePrediction(
    edge_embedding_methods="Concatenate",
    use_scale_free_distribution=True,
    training_unbalance_rate=1,
    max_depth=100,
    n_estimators=100,
)
embedder_node2vec_bfs = Node2VecSkipGramEnsmallen(
    random_state=seed,
    return_weight=5,
    explore_weight=0.2,
    embedding_size=10,
    verbose=False,
    enable_cache=True,
)

In [ ]:
import importlib

importlib.reload(helper_lib.predict)

pairs_to_predict = [
    ("miRNA", "Phenotype"),
    ("lncRNA", "Phenotype"),
    ("miRNA", "Gene"),
    ("miRNA", "GO"),
    ("lncRNA", "GO"),
    ("lncRNA", "Disease"),
    ("Protein", "GO"),
]

results_fun_node2vecBFS_tree = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_node2vec_bfs,
    model_tree,
    pairs_to_predict,
    seed=seed,
    clear_output=False,
    use_scale_free_distribution=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
)
results_fun_node2vecBFS_tree.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_tree_f1_2.csv"
)
results_fun_node2vecBFS_forest = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_node2vec_bfs,
    model_forest,
    pairs_to_predict,
    seed=seed,
    clear_output=False,
    use_scale_free_distribution=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
)
results_fun_node2vecBFS_forest.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_forest_f1_2.csv"
)

full_results = pd.concat([results_fun_node2vecBFS_tree, results_fun_node2vecBFS_forest])
full_results.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_node2vecBFS_f1_2.csv"
)

In [ ]:
full_results = pd.read_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_node2vecBFS_f1_2.csv"
)

for elem in full_results["name"].unique():
    for pair in pairs_to_predict:
        df_results = full_results[full_results["name"] == elem]
        model = "".join(df_results["Model"].iloc[0].split(" ")[:2])
        results_custom_model = df_results[
            (df_results["Source Type"] == pair[0])
            & (df_results["Destination Type"] == pair[1])
        ]
        positive = results_custom_model["Positive balanced accuracy"]
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%", "\\%")
        negative = results_custom_model["Negative balanced accuracy"]
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%", "\\%")
        mean = results_custom_model["Mean balanced accuracy"]
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%", "\\%")
        f1 = results_custom_model["F1-Binary"]
        f1_print = f"{f1.mean():.2%}±{f1.std():.2%}".replace("%", "\\%")
        print(
            f"{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} & {f1_print} \\\\"
        )

### Unbalanced

In [ ]:
results_fun_node2vecBFS_tree_unbalanced = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_node2vec_bfs,
    model_tree,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
    testing_unbalance_rate=10,
)
results_fun_node2vecBFS_tree_unbalanced.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_tree_f1_2_unbalanced.csv"
)
results_fun_node2vecBFS_forest_unbalanced = helper_lib.predict.edge_pred_pairs(
    view_undirected_rnakg,
    embedder_node2vec_bfs,
    model_forest,
    pairs_to_predict,
    seed=seed,
    clear_output=True,
    save_embedding_path=f"./RNA-KG_notebooks/Views/view{view_number}/embeddings/",
    save_model_path=f"./RNA-KG_notebooks/Views/view{view_number}/models/",
    testing_unbalance_rate=10,
)
results_fun_node2vecBFS_forest_unbalanced.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/results_fun_node2vecBFS_forest_f1_2_unbalanced.csv"
)

full_results_unabalanced_n2v = pd.concat(
    [results_fun_node2vecBFS_tree_unbalanced, results_fun_node2vecBFS_forest_unbalanced]
)
full_results_unabalanced_n2v.to_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_node2vecBFS_f1_2_unbalanced.csv"
)

In [ ]:
full_results_unabalanced_n2v = pd.read_csv(
    f"./RNA-KG_notebooks/Views/view{view_number}/csv/result_custom_filtered_final_node2vecBFS_f1_2_unbalanced.csv"
)

for elem in full_results_unabalanced_n2v["name"].unique():
    for pair in pairs_to_predict:
        df_results = full_results_unabalanced_n2v[
            full_results_unabalanced_n2v["name"] == elem
        ]
        model = "".join(df_results["Model"].iloc[0].split(" ")[:2])
        results_custom_model = df_results[
            (df_results["Source Type"] == pair[0])
            & (df_results["Destination Type"] == pair[1])
        ]
        positive = results_custom_model["Positive balanced accuracy"]
        pos_print = f"{positive.mean():.2%}±{positive.std():.2%}".replace("%", "\\%")
        negative = results_custom_model["Negative balanced accuracy"]
        neg_print = f"{negative.mean():.2%}±{negative.std():.2%}".replace("%", "\\%")
        mean = results_custom_model["Mean balanced accuracy"]
        mean_print = f"{mean.mean():.2%}±{mean.std():.2%}".replace("%", "\\%")
        f1 = results_custom_model["F1-Binary"]
        f1_print = f"{f1.mean():.2%}±{f1.std():.2%}".replace("%", "\\%")
        print(
            f"{model} & {pair[0]}-{pair[1]} & {pos_print} & {neg_print} & {mean_print} & {f1_print}\\\\"
        )